In [29]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    tool,
    executor,
)
from agent_framework.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


✅ All imports successful!


## 第一步：定义用于结构化输出的 Pydantic 模型

这些模型定义了代理将返回的**模式**。使用 `response_format` 和 Pydantic 可以确保：
- ✅ 类型安全的数据提取
- ✅ 自动验证
- ✅ 无需从自由文本响应中解析错误
- ✅ 基于字段的轻松条件路由


In [30]:
# 定义 `BookingCheckResult` 类，用来封装一组相关的数据或行为。
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


# 定义 `AlternativeResult` 类，用来封装一组相关的数据或行为。
class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


# 定义 `BookingConfirmation` 类，用来封装一组相关的数据或行为。
class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")


✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## 第2步：创建酒店预订工具

这个工具是 **availability_agent** 用来检查房间是否可用的。我们使用 `@tool` 装饰器来：
- 将 Python 函数转换为 AI 可调用工具
- 自动为 LLM 生成 JSON schema
- 处理参数验证
- 允许代理自动调用

在这个演示中：
- **斯德哥尔摩、西雅图、东京、伦敦、阿姆斯特丹** → 有房间 ✅
- **其他所有城市** → 无房间 ❌


In [31]:
# 使用装饰器为下面的函数或类添加框架能力。
@tool(description="Check hotel room availability for a destination city")
# 定义函数 `hotel_booking`，把一段可复用逻辑封装起来。
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    # 把字符串统一转成小写，便于做不区分大小写的比较。
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    # 返回当前函数的结果给调用方。
    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")


✅ hotel_booking tool created with @tool decorator


## 第三步：定义用于路由的条件函数

这些函数会检查代理的响应，并决定在工作流中采取哪条路径。

**关键模式：**
1. 检查消息是否为 `AgentExecutorResponse`
2. 解析结构化输出（Pydantic 模型）
3. 返回 `True` 或 `False` 来控制路由

工作流将在**边缘**上评估这些条件，以决定接下来调用哪个执行器。


In [32]:
# 定义一个辅助函数：从可能是列表的 payload 里取出最关键的那一项。
def unwrap_payload(payload: Any) -> Any:
    # 如果当前 payload 是列表，就优先取第一个元素继续解析。
    if isinstance(payload, list) and payload:
        return unwrap_payload(payload[0])
    # 否则就直接返回原对象。
    return payload


# 定义一个辅助函数：尽量从不同响应对象里提取可解析的 JSON 文本。
def extract_response_text(payload: Any) -> str | None:
    # 先把列表形式的 payload 展开成核心对象。
    payload = unwrap_payload(payload)

    # 先尝试最常见的 text 字段。
    direct_text = getattr(payload, "text", None)
    if isinstance(direct_text, str) and direct_text.strip():
        return direct_text

    # 当前版本的 AgentExecutorResponse 使用 agent_response 字段。
    agent_response = getattr(payload, "agent_response", None)
    nested_text = getattr(agent_response, "text", None)
    if isinstance(nested_text, str) and nested_text.strip():
        return nested_text

    # 兼容另一类命名：agent_run_response。
    agent_run_response = getattr(payload, "agent_run_response", None)
    legacy_text = getattr(agent_run_response, "text", None)
    if isinstance(legacy_text, str) and legacy_text.strip():
        return legacy_text

    # 如果对象带有 messages，就优先取最后一条文本消息。
    messages = getattr(payload, "messages", None)
    if isinstance(messages, list):
        for message in reversed(messages):
            message_text = getattr(message, "text", None)
            if isinstance(message_text, str) and message_text.strip():
                return message_text

    # 如果 agent_response 里也有 messages，同样尝试最后一条。
    nested_messages = getattr(agent_response, "messages", None)
    if isinstance(nested_messages, list):
        for message in reversed(nested_messages):
            message_text = getattr(message, "text", None)
            if isinstance(message_text, str) and message_text.strip():
                return message_text

    # 兼容 full_conversation 这种字段。
    full_conversation = getattr(payload, "full_conversation", None)
    if isinstance(full_conversation, list):
        for message in reversed(full_conversation):
            message_text = getattr(message, "text", None)
            if isinstance(message_text, str) and message_text.strip():
                return message_text

    return None


# 定义函数 `has_availability_condition`，把一段可复用逻辑封装起来。
def has_availability_condition(message: Any) -> bool:
    """当有房间时返回 True。"""
    try:
        raw_text = extract_response_text(message)
        if not raw_text:
            return False

        result = BookingCheckResult.model_validate_json(raw_text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️ Condition Parse Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


# 定义函数 `no_availability_condition`，把一段可复用逻辑封装起来。
def no_availability_condition(message: Any) -> bool:
    """当没有房间时返回 True。"""
    try:
        raw_text = extract_response_text(message)
        if not raw_text:
            return False

        result = BookingCheckResult.model_validate_json(raw_text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability = <strong>{not result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️ Condition Parse Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


print("✅ Condition functions defined:")
print("   - unwrap_payload (normalize list payloads)")
print("   - extract_response_text (normalize response objects)")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")


✅ Condition functions defined:
   - unwrap_payload (normalize list payloads)
   - extract_response_text (normalize response objects)
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)


## 第四步：创建自定义显示执行器

执行器是执行转换或副作用的工作流组件。我们使用 `@executor` 装饰器来创建一个自定义执行器，用于显示最终结果。

**关键概念：**
- `@executor(id="...")` - 将函数注册为工作流执行器
- `WorkflowContext[Never, str]` - 输入/输出的类型提示
- `ctx.yield_output(...)` - 生成最终的工作流结果


In [33]:
# 使用装饰器为下面的函数或类添加框架能力。
@executor(id="display_result")
# 定义异步函数 `display_result`，用于处理需要 `await` 的流程。
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    raw_text = extract_response_text(response)
    if raw_text:
        await ctx.yield_output(raw_text)
    else:
        await ctx.yield_output(str(response))


# 定义一个辅助函数：把不同类型的输出对象尽量提取成 JSON 文本候选项。
def extract_json_candidates(output: object) -> list[str]:
    candidates: list[str] = []
    raw_text = extract_response_text(output)
    if isinstance(raw_text, str) and raw_text.strip():
        candidates.append(raw_text)

    fallback = str(output)
    if fallback.strip():
        candidates.append(fallback)

    deduped: list[str] = []
    for item in candidates:
        if item not in deduped:
            deduped.append(item)
    return deduped


# 定义一个辅助函数：从 outputs 里挑出第一个能匹配目标模型的 JSON。
def pick_structured_output(outputs: list[object], model: type[BaseModel]) -> BaseModel:
    for output in reversed(outputs):
        for candidate in extract_json_candidates(output):
            try:
                return model.model_validate_json(candidate)
            except Exception:
                continue
    raise ValueError(f"No output matched model {model.__name__}: {outputs}")


# 定义一个辅助函数：从事件流中按 executor_id 精准提取结构化结果。
def pick_model_from_events(events: list[object], model: type[BaseModel], preferred_executor_ids: list[str]) -> BaseModel:
    for event in reversed(events):
        executor_id = getattr(event, "executor_id", None)
        if executor_id not in preferred_executor_ids:
            continue
        for candidate in extract_json_candidates(getattr(event, "data", None)):
            try:
                return model.model_validate_json(candidate)
            except Exception:
                continue

    for event in reversed(events):
        for candidate in extract_json_candidates(getattr(event, "data", None)):
            try:
                return model.model_validate_json(candidate)
            except Exception:
                continue

    raise ValueError(f"No event matched model {model.__name__}. Events: {events}")


print("✅ display_result executor created with @executor decorator")
print("✅ pick_structured_output helper created")
print("✅ pick_model_from_events helper created")


✅ display_result executor created with @executor decorator
✅ pick_structured_output helper created
✅ pick_model_from_events helper created


## 第五步：加载环境变量

配置 LLM 客户端。本示例适用于以下情况：
- **GitHub 模型**（使用 GitHub 令牌的免费版本）
- **Azure OpenAI**
- **OpenAI**


In [34]:
# Load environment variables
# 从 `.env` 文件加载环境变量配置。
load_dotenv()

# Configure DashScope Qwen via the OpenAI-compatible Chat Completions client
# 创建兼容 OpenAI 接口的聊天客户端，用来连接模型服务。
chat_client = OpenAIChatCompletionClient(
    base_url="https://models.inference.ai.azure.com/",  # DashScope OpenAI兼容接口
    api_key=os.environ.get("GITHUB_TOKEN"),                  # DashScope API Key
    model="gpt-4o-mini"                                              # 使用的模型名称
)

print("✅ Chat client configured with DashScope qwen-max")


✅ Chat client configured with DashScope qwen-max


## 第六步：创建具有结构化输出的AI代理

我们创建了**三个专门的代理**，每个代理都封装在一个 `AgentExecutor` 中：

1. **availability_agent** - 使用工具检查酒店可用性  
2. **alternative_agent** - 在没有房间时建议替代城市  
3. **booking_agent** - 在有房间时鼓励预订  

**主要特点：**  
- `tools=[hotel_booking]` - 为代理提供工具  
- `response_format=PydanticModel` - 强制生成结构化的JSON输出  
- `AgentExecutor(..., id="...")` - 将代理封装以供工作流使用  


In [35]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    # 创建一个具体的 Agent，并配置它的职责和输出格式。
    Agent(
        client=chat_client,
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status. "
            "You MUST return valid JSON only."
        ),
        name="availability_agent",
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    # 创建一个具体的 Agent，并配置它的职责和输出格式。
    Agent(
        client=chat_client,
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful. "
            "You MUST return valid JSON only."
        ),
        name="alternative_agent",
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    # 创建一个具体的 Agent，并配置它的职责和输出格式。
    Agent(
        client=chat_client,
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging. "
            "You MUST return valid JSON only."
        ),
        name="booking_agent",
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## 第七步：使用条件边构建工作流

现在我们使用 `WorkflowBuilder` 来构建具有条件路由的图：

**工作流结构：**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**关键方法：**
- `.set_start_executor(...)` - 设置入口点
- `.add_edge(from, to, condition=...)` - 添加条件边
- `.build()` - 完成工作流构建


In [36]:
# Build the workflow with conditional routing
workflow = (
    # 创建基础工作流构建器，用来编排节点之间的流转关系。
    WorkflowBuilder(start_executor=availability_agent)
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    # 根据前面配置生成最终可运行的工作流对象。
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)


## 第8步：运行测试用例1 - 无可用房间的城市（巴黎）

让我们通过请求巴黎的酒店（在我们的模拟中没有房间）来测试**无可用性**路径。


In [37]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

request_paris = AgentExecutorRequest(
    messages=[Message("user", ["I want to book a hotel in Paris"])],
    should_respond=True,
)

# 使用当前版本支持的 stream=True 接口收集完整事件流。
stream_paris = workflow.run(request_paris, stream=True)
# 逐个消费事件，保留所有中间节点信息。
events_paris = [event async for event in stream_paris]
# 读取这次流式运行的最终结果对象。
result_paris_run = await stream_paris.get_final_response()
# 把最终输出也拿出来，作为事件扫描失败时的兜底来源。
outputs_paris = result_paris_run.get_outputs()

try:
    # 优先从事件流里提取 alternative_agent / display_result 的结果。
    result_paris = pick_model_from_events(
        events_paris,
        AlternativeResult,
        preferred_executor_ids=["alternative_agent", "display_result"],
    )
except Exception:
    # 如果事件流里没拿到，再回退到最终输出列表中继续解析。
    result_paris = pick_structured_output(outputs_paris, AlternativeResult)

display(
    HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
        <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
        <div style='background: white; padding: 20px; border-radius: 8px;'>
            <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
            <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
            <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
        </div>
    </div>
""")
)


## 第9步：运行测试用例2 - 有可用房间的城市（斯德哥尔摩）

现在，让我们通过请求斯德哥尔摩的酒店（在我们的模拟中有房间）来测试**可用性**路径。


In [38]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

request_stockholm = AgentExecutorRequest(
    messages=[Message("user", ["I want to book a hotel in Stockholm"])],
    should_respond=True,
)

# 使用当前版本支持的 stream=True 接口收集完整事件流。
stream_stockholm = workflow.run(request_stockholm, stream=True)
# 逐个消费事件，保留所有中间节点信息。
events_stockholm = [event async for event in stream_stockholm]
# 读取这次流式运行的最终结果对象。
result_stockholm_run = await stream_stockholm.get_final_response()
# 把最终输出也拿出来，作为事件扫描失败时的兜底来源。
outputs_stockholm = result_stockholm_run.get_outputs()

try:
    # 优先从事件流里提取 booking_agent / display_result 的结果。
    result_stockholm = pick_model_from_events(
        events_stockholm,
        BookingConfirmation,
        preferred_executor_ids=["booking_agent", "display_result"],
    )
except Exception:
    # 如果事件流里没拿到，再回退到最终输出列表中继续解析。
    result_stockholm = pick_structured_output(outputs_stockholm, BookingConfirmation)

display(
    HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
        <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
        <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
            <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
            <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
            <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
            <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
        </div>
    </div>
""")
)


## 关键要点和下一步计划

### ✅ 您学到了什么：

1. **WorkflowBuilder 模式**
   - 使用 `.set_start_executor()` 定义入口点
   - 使用 `.add_edge(from, to, condition=...)` 设置条件路由
   - 调用 `.build()` 完成工作流构建

2. **条件路由**
   - 条件函数检查 `AgentExecutorResponse`
   - 解析结构化输出以做出路由决策
   - 返回 `True` 激活边，返回 `False` 跳过边

3. **工具集成**
   - 使用 `@tool` 将 Python 函数转换为 AI 工具
   - 代理在需要时自动调用工具
   - 工具返回 JSON，代理可以解析这些数据

4. **结构化输出**
   - 使用 Pydantic 模型进行类型安全的数据提取
   - 创建代理时设置 `response_format=MyModel`
   - 使用 `Model.model_validate_json()` 解析响应

5. **自定义执行器**
   - 使用 `@executor(id="...")` 创建工作流组件
   - 执行器可以转换数据或执行副作用
   - 使用 `ctx.yield_output()` 生成工作流结果

### 🚀 实际应用场景：

- **旅行预订**：检查可用性，建议替代方案，比较选项
- **客户服务**：根据问题类型、情绪、优先级进行路由
- **电子商务**：检查库存，建议替代品，处理订单
- **内容审核**：根据毒性评分、用户标记进行路由
- **审批工作流**：根据金额、用户角色、风险级别进行路由
- **多阶段处理**：根据数据质量、完整性进行路由

### 📚 下一步计划：

- 添加更复杂的条件（多重标准）
- 使用工作流状态管理实现循环
- 添加子工作流以实现可复用组件
- 与真实 API 集成（如酒店预订、库存系统）
- 添加错误处理和备用路径
- 使用内置可视化工具展示工作流



---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
